In [1]:
import requests

def get_product_data(barcode):
    """
    Obtiene datos de un producto desde la API v2 de Open Food Facts.
    """
    # 1. Configurar el User-Agent (REQUISITO CRÍTICO DE LA API)
    headers = {
        'User-Agent': 'MyDataScienceProject/1.0 (mklanibarro@gmail.com)'
    }
    
    # 2. Construir la URL de la API v2
    url = f"https://world.openfoodfacts.org/api/v2/product/{barcode}.json"
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            
            if data.get('status') == 1: # 1 significa producto encontrado
                product = data['product']
                
                # 3. Extraer solo lo que necesitamos para el ML
                result = {
                    'nombre': product.get('product_name', 'Desconocido'),
                    'nutriscore': product.get('nutriscore_grade', 'N/A'),
                    'nova': product.get('nova_group', 'N/A'),
                    # Estos son los tags de aditivos ya procesados por su IA
                    'aditivos_lista': product.get('additives_tags', []),
                    'conteo_aditivos': len(product.get('additives_tags', [])),
                    'ingredientes': product.get('ingredients_text', ''),
                    'categorias': product.get('categories_hierarchy', [])
                }
                return result
            else:
                return "Producto no encontrado en la base de datos."
        else:
            return f"Error en la petición: {response.status_code}"
            
    except Exception as e:
        return f"Error de conexión: {e}"

# Ejemplo de uso con un código de barras real (ej. Nutella)
# print(get_product_data("3017620422003"))

In [2]:
get_product_data("8410076801180")

{'nombre': 'Crunchy oats&honey',
 'nutriscore': 'd',
 'nova': 4,
 'aditivos_lista': ['en:e322', 'en:e500', 'en:e500ii'],
 'conteo_aditivos': 3,
 'ingredientes': 'Avoine complète (60%), sucre, huiles végétales (tournesol, colza en proportions variables), miel (3%), sel de cuisine, mélasse, émulsifiant (lécithines), poudre à lever (bicarbonate de sodium).',
 'categorias': ['en:plant-based-foods-and-beverages',
  'en:plant-based-foods',
  'en:snacks',
  'en:cereals-and-potatoes',
  'en:sweet-snacks',
  'en:bars',
  'en:cereal-bars']}

In [3]:
import requests
import pandas as pd

def generar_maestro_aditivos():
    url = "https://world.openfoodfacts.org/data/taxonomies/additives.json"
    headers = {'User-Agent': 'MyDataScienceProject/1.0 (mklanibarro@gmail.com)'}
    
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        data = response.json()
        
        lista_limpia = []
        for key, value in data.items():
            # Solo nos interesan los que tienen nombre en español o inglés
            nombre = value.get('name', {}).get('en', value.get('name', {}).get('en', 'Desconocido'))
            
            lista_limpia.append({
                'id': key,
                'nombre': nombre
            })
        
        df = pd.DataFrame(lista_limpia)
        df.to_csv('./maestro_aditivos.csv', index=False)
        print(f"¡Hecho! Tienes {len(df)} aditivos registrados.")
        return df

# Ejecutar una vez para tener tu archivo local
# df_aditivos = generar_maestro_aditivos()

In [4]:
df_aditivos = generar_maestro_aditivos()

¡Hecho! Tienes 681 aditivos registrados.


In [5]:
df_aditivos.head()

,id,nombre
0,en:e333ii,E333ii - Dicalcium citrate
1,en:e321,E321 - Butylated hydroxytoluene
2,en:e345,E345 - Magnesium citrate
3,en:e303,E303 - Potassium ascorbate
4,en:e1104,E1104 - lipase


In [6]:
import pandas as pd
import re

def procesar_maestro():
    df = pd.read_csv('maestro_aditivos.csv')
    
    # Extraemos el código E (ej: E321) y el nombre limpio
    # Usamos regex para mayor seguridad
    df['codigo_e'] = df['nombre'].str.extract(r'(E\d+[a-z]*)', flags=re.IGNORECASE)
    df['nombre_cientifico'] = df['nombre'].apply(lambda x: x.split(' - ')[-1] if ' - ' in str(x) else x)
    
    # Limpiamos espacios y posibles duplicados
    df['nombre_cientifico'] = df['nombre_cientifico'].str.strip().str.lower()
    
    return df

df_limpio = procesar_maestro()

In [7]:
import requests

def clasificar_evidencia(componente):
    base_url = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
    
    # Query balanceada: busca el aditivo asociado a términos de salud
    query = f'"{componente}" AND (health OR pathology OR benefit OR risk)'
    
    params = {
        'query': query,
        'format': 'json',
        'pageSize': 1, # Solo queremos el hitCount
        'resultType': 'lite'
    }
    
    try:
        response = requests.get(base_url, params=params)
        data = response.json()
        n_papers = int(data.get('hitCount', 0))
        
        # Lógica de clasificación (puedes ajustarla según tu criterio)
        if n_papers > 500:
            status = "Alta"
        elif 50 <= n_papers <= 500:
            status = "Media"
        else:
            status = "Frágil"
            
        return status, n_papers
    except:
        return "Error", 0

# Prueba con un ejemplo concreto
# status, count = clasificar_evidencia("aspartame")

In [ ]:
import requests

def analizar_abstract(nombre_aditivo):
    base_url = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
    # Buscamos en título y resumen específicamente
    query = f'(TITLE:"{nombre_aditivo}" OR ABSTRACT:"{nombre_aditivo}")'
    
    params = {
        'query': query,
        'format': 'json',
        'pageSize': 25, # Analizamos los 25 más relevantes
        'resultType': 'core' # 'core' devuelve el abstract completo
    }
    
    response = requests.get(base_url, params=params)
    results = response.json().get('resultList', {}).get('result', [])
    
    # Listas de términos para clasificar el "sentimiento científico"
    keywords_riesgo = [
    # Toxicología general
    'toxic', 'toxicity', 'cytotoxicity', 'genotoxicity', 'mutagenic', 'carcinogenic',
    # Inflamación y estrés
    'inflammation', 'pro-inflammatory', 'oxidative stress', 'cytokine', 'interleukin',
    # Sistema endocrino y metabólico
    'endocrine disruptor', 'hormonal interference', 'metabolic syndrome', 'obesogenic',
    'insulin resistance', 'hyperglycemia', 'dyslipidemia',
    # Alergias y sensibilidad
    'allergen', 'hypersensitivity', 'intolerance', 'anaphylaxis', 'immunogenicity',
    # Órganos específicos
    'hepatotoxic', 'nephrotoxic', 'neurotoxic', 'gastrointestinal distress',
    # Evidencia epidemiológica
    'risk factor', 'adverse effect', 'lethal dose', 'noael', 'loael', 'chronic exposure'
]
    keywords_beneficio = [
    # Actividad antioxidante y protectora
    'antioxidant', 'free radical scavenger', 'protective', 'cytoprotective', 'neuroprotective',
    'cardioprotective', 'chemopreventive',
    # Nutrición y fisiología
    'essential', 'vitamin', 'micronutrient', 'bioavailable', 'bioactive', 'nutrient',
    # Salud digestiva
    'prebiotic', 'probiotic', 'gut microbiota', 'microbiome health', 'digestibility',
    # Efectos metabólicos positivos
    'anti-inflammatory', 'hypoglycemic', 'lipid-lowering', 'metabolic health',
    # Seguridad y bienestar
    'safe', 'generally recognized as safe', 'gras', 'therapeutic', 'benefit', 'healthy',
    'fortification', 'enrichment'
]
    
    puntos_riesgo = 0
    puntos_beneficio = 0
    
    for article in results:
        # Combinamos título y abstract para el análisis
        texto_analisis = (article.get('title', '') + " " + article.get('abstractText', '')).lower()
        
        # Conteo simple
        puntos_riesgo += sum(1 for word in keywords_riesgo if word in texto_analisis)
        puntos_beneficio += sum(1 for word in keywords_beneficio if word in texto_analisis)
    
    # Decisión final del aditivo
    if puntos_beneficio > puntos_riesgo:
        return "Positivo/Nutricional", puntos_beneficio
    elif puntos_riesgo > 0:
        return "Potencial Riesgo", puntos_riesgo
    else:
        return "Neutro/Poca evidencia", 0

In [9]:
import re

def analizar_sentimiento_cientifico(abstract):
    # Convertimos a minúsculas para normalizar
    texto = abstract.lower()
    
    # Creamos patrones regex para capturar raíces de palabras
    # El \b asegura que coincida el inicio de la palabra
    patron_riesgo = re.compile(r'\b(' + '|'.join(keywords_riesgo) + r')')
    patron_beneficio = re.compile(r'\b(' + '|'.join(keywords_beneficio) + r')')
    
    # Contamos apariciones únicas o totales
    conteo_riesgo = len(patron_riesgo.findall(texto))
    conteo_beneficio = len(patron_beneficio.findall(texto))
    
    return conteo_riesgo, conteo_beneficio

In [10]:
def calcular_perfil_salud(conteo_beneficio, conteo_riesgo):
    # Aplicamos la fórmula del ratio
    ratio = (conteo_beneficio + 1) / (conteo_riesgo + 1)
    
    # Clasificación lógica
    if ratio > 2.0:
        categoria = "Beneficioso / Antioxidante"
        color = "green"
    elif ratio < 0.5:
        categoria = "Estudiar con Precaución (Riesgo)"
        color = "red"
    else:
        categoria = "Neutro / Uso Tecnológico"
        color = "gray"
        
    return ratio, categoria

# Ejemplo de uso con datos hipotéticos
# Supongamos que E300 (Ácido Ascórbico) devuelve: beneficio=45, riesgo=2
ratio_e300, cat_e300 = calcular_perfil_salud(45, 2)
print(f"E300 - Ratio: {ratio_e300:.2f} | Categoría: {cat_e300}")

# Supongamos que un colorante polémico devuelve: beneficio=1, riesgo=12
ratio_col, cat_col = calcular_perfil_salud(1, 12)
print(f"Colorante - Ratio: {ratio_col:.2f} | Categoría: {cat_col}")

E300 - Ratio: 15.33 | Categoría: Beneficioso / Antioxidante
Colorante - Ratio: 0.15 | Categoría: Estudiar con Precaución (Riesgo)


In [11]:
# Diccionario de ingredientes con impacto positivo
keywords_noble = [
    # Aceites de alta calidad
    'olive oil', 'extra virgin', 'avocado oil', 'flaxseed oil',
    # Extractos y plantas
    'extract', 'curcuma', 'turmeric', 'ginger', 'rosemary', 'green tea',
    'aloe vera', 'echinacea', 'ginseng', 'fruit extract',
    # Superalimentos/Semillas
    'chia', 'quinoa', 'kale', 'spirulina', 'chlorella', 'hemp',
    # Otros
    'probiotic', 'omega-3', 'polyphenols', 'flavonoids'
]

In [12]:
def analizar_producto_completo(barcode):
    data = get_product_data(barcode) # Tu función existente
    if isinstance(data, str): return data
    
    texto_ingredientes = data.get('ingredientes', '').lower()
    
    # 1. Contamos ingredientes positivos (noble ingredients)
    conteo_noble = sum(1 for ing in keywords_noble if ing in texto_ingredientes)
    
    # 2. Procesamos aditivos (tu lógica actual)
    aditivos = data.get('aditivos_lista', [])
    
    # Aquí es donde el modelo de ML entra en juego
    # El modelo recibirá: [conteo_aditivos, ratio_evidencia_avg, conteo_noble]
    return {
        'nombre': data['nombre'],
        'puntos_nobles': conteo_noble,
        'n_aditivos': len(aditivos)
    }

In [13]:
import requests

def consulta_pubchem(nombre_quimico):
    # La API de PubChem permite buscar por nombre y pedir propiedades específicas
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{nombre_quimico}/property/MolecularWeight,XLogP/JSON"
    
    try:
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            data = response.json()
            propiedades = data['PropertyTable']['Properties'][0]
            return {
                'peso_molecular': propiedades.get('MolecularWeight'),
                'logP': propiedades.get('XLogP')
            }
    except:
        return {'peso_molecular': None, 'logP': None}

# Ejemplo: consulta_pubchem("Ascorbic acid")

In [14]:
consulta_pubchem("Ascorbic acid")

{'peso_molecular': '176.12', 'logP': -1.6}

In [15]:
import time
import pandas as pd

df_maestro = pd.read_csv('maestro_aditivos.csv') #[cite: 1]

resultados_enriquecidos = []

# Procesamos una muestra para no saturar las APIs
for index, row in df_maestro.head(10).iterrows():
    nombre = row['nombre'].split(' - ')[-1] # Extraemos el nombre químico[cite: 1]
    
    # 1. Datos Científicos (Función que definimos antes)
    evidencia = analizar_abstract(nombre) 
    
    # 2. Datos Químicos
    quimica = consulta_pubchem(nombre)
    
    # Combinamos todo en un diccionario
    registro = {
        'id': row['id'], #[cite: 1]
        'nombre': nombre,
        'evidencia_ratio': evidencia[1],
        'clase_evidencia': evidencia[0],
        'peso_molecular': quimica['peso_molecular'],
        'logP': quimica['logP']
    }
    resultados_enriquecidos.append(registro)
    
    # Respetamos los límites de la API
    time.sleep(0.5)

df_final = pd.DataFrame(resultados_enriquecidos)

TypeError: 'NoneType' object is not subscriptable

In [16]:
import requests
import xml.etree.ElementTree as ET
import time

def consultar_pubmed(aditivo, max_results=5):
    """
    Busca un aditivo en PubMed y devuelve títulos y abstracts.
    """
    base_url_search = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    base_url_fetch = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    
    # Paso 1: Buscar IDs de artículos relacionados
    # Buscamos el aditivo junto con términos de salud para filtrar mejor
    query = f"{aditivo} AND (health OR safety OR toxicity)"
    search_params = {
        "db": "pubmed",
        "term": query,
        "retmode": "json",
        "retmax": max_results
    }
    
    try:
        search_res = requests.get(base_url_search, params=search_params)
        id_list = search_res.json().get("esearchresult", {}).get("idlist", [])
        
        if not id_list:
            return []

        # Paso 2: Obtener los detalles (Abstracts) de esos IDs
        fetch_params = {
            "db": "pubmed",
            "id": ",".join(id_list),
            "retmode": "xml",
            "rettype": "abstract"
        }
        
        fetch_res = requests.get(base_url_fetch, params=fetch_params)
        root = ET.fromstring(fetch_res.content)
        
        resultados = []
        for article in root.findall(".//PubmedArticle"):
            titulo = article.find(".//ArticleTitle").text if article.find(".//ArticleTitle") is not None else "Sin título"
            
            # El abstract puede estar dividido en varias secciones (Label)
            abstract_texts = article.findall(".//AbstractText")
            abstract_completo = " ".join([parts.text for parts in abstract_texts if parts.text is not None])
            
            resultados.append({
                "titulo": titulo,
                "abstract": abstract_completo
            })
            
        return resultados

    except Exception as e:
        print(f"Error consultando PubMed para {aditivo}: {e}")
        return []

# Ejemplo de uso:
# info = consultar_pubmed("Aspartame", max_results=3)
# for art in info:
#     print(f"Título: {art['titulo']}\n")

In [17]:
info = consultar_pubmed("Aspartame", max_results=3)
for art in info:
    print(f"Título: {art['titulo']}\n")

Título: The Safety Profile of Aspartame: A Review of Regulatory Standards and Emerging Health Concerns.

Título: Odorous disinfection byproducts from sweet peptides during chlorination.

Título: Long-term aspartame consumption affects global levels of DNA methylation in different types of rat tissues.



In [18]:
aditivo_test = "aspartame"
articulos = consultar_pubmed(aditivo_test)

total_riesgo = 0
total_beneficio = 0

for art in articulos:
    # Usas tu función de regex sobre el abstract de PubMed
    r, b = analizar_sentimiento_cientifico(art['abstract'])[cite: 2]
    total_riesgo += r
    total_beneficio += b

ratio, categoria = calcular_perfil_salud(total_beneficio, total_riesgo)[cite: 2]
print(f"Resultado para {aditivo_test}: {categoria} (Ratio: {ratio:.2f})")

NameError: name 'keywords_riesgo' is not defined

In [26]:
import requests
import xml.etree.ElementTree as ET
import time
import re

def analizar_aditivo_pubmed(aditivo):
    # 1. Definición de diccionarios (dentro para evitar el error 'not defined')
    keywords_riesgo = [
        'toxic', 'toxicity', 'cytotoxicity', 'genotoxicity', 'mutagenic', 'carcinogenic',
        'inflammation', 'pro-inflammatory', 'oxidative stress', 'cytokine', 'interleukin',
        'endocrine disruptor', 'hormonal interference', 'metabolic syndrome', 'obesogenic',
        'insulin resistance', 'hyperglycemia', 'dyslipidemia', 'allergen', 'hypersensitivity', 
        'intolerance', 'anaphylaxis', 'immunogenicity', 'hepatotoxic', 'nephrotoxic', 
        'neurotoxic', 'gastrointestinal distress', 'risk factor', 'adverse effect', 
        'lethal dose', 'noael', 'loael', 'chronic exposure'
    ]
    
    keywords_beneficio = [
        'antioxidant', 'free radical scavenger', 'protective', 'cytoprotective', 'neuroprotective',
        'cardioprotective', 'chemopreventive', 'essential', 'vitamin', 'micronutrient', 
        'bioavailable', 'bioactive', 'nutrient', 'prebiotic', 'probiotic', 'gut microbiota', 
        'microbiome health', 'digestibility', 'anti-inflammatory', 'hypoglycemic', 
        'lipid-lowering', 'metabolic health', 'safe', 'gras', 'therapeutic', 'benefit', 
        'healthy', 'fortification', 'enrichment'
    ]

    # 2. Configuración de la API
    base_url_search = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    base_url_fetch = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    
    query = f'"{aditivo}" AND (health OR safety OR toxicity OR benefit)'
    
    try:
        # Paso 1: Buscar IDs
        search_params = {"db": "pubmed", "term": query, "retmode": "json", "retmax": 10}
        search_res = requests.get(base_url_search, params=search_params, timeout=10)
        data_search = search_res.json()
        
        id_list = data_search.get("esearchresult", {}).get("idlist", [])
        total_papers = int(data_search.get("esearchresult", {}).get("count", 0))
        
        if not id_list:
            return {"total_encontrados": total_papers, "menciones_positivas": 0, "menciones_negativas": 0, "ratio_evidencia": 1.0, "clasificacion": "Sin datos"}

        # Paso 2: Obtener y analizar Abstracts
        fetch_params = {"db": "pubmed", "id": ",".join(id_list), "retmode": "xml"}
        fetch_res = requests.get(base_url_fetch, params=fetch_params, timeout=15)
        root = ET.fromstring(fetch_res.content)
        
        conteo_riesgo = 0
        conteo_beneficio = 0
        
        for article in root.findall(".//PubmedArticle"):
            titulo = article.find(".//ArticleTitle").text or ""
            abstract_texts = article.findall(".//AbstractText")
            abstract_completo = " ".join([p.text for p in abstract_texts if p.text is not None])
            
            texto_full = (titulo + " " + abstract_completo).lower()
            
            # Conteo usando regex para mayor precisión
            conteo_riesgo += len(re.findall(r'\b(' + '|'.join(keywords_riesgo) + r')', texto_full))
            conteo_beneficio += len(re.findall(r'\b(' + '|'.join(keywords_beneficio) + r')', texto_full))
            
        # 3. Cálculo de métricas finales
        ratio = (conteo_beneficio + 1) / (conteo_riesgo + 1)
        
        if ratio > 2.0:
            categoria = "Positivo/Beneficioso"
        elif ratio < 0.5:
            categoria = "Negativo/Riesgo"
        else:
            categoria = "Neutro/Tecnológico"
            
        return {
            "total_encontrados": total_papers,
            "menciones_positivas": conteo_beneficio,
            "menciones_negativas": conteo_riesgo,
            "ratio_evidencia": round(ratio, 2),
            "clasificacion": categoria
        }

    except Exception as e:
        print(f"Error procesando {aditivo}: {e}")
        return None


In [24]:
# Ejemplo de ejecución:
resultado = analizar_aditivo_pubmed("Aspartame")
print(resultado)

{'total_encontrados': 793, 'menciones_positivas': 0, 'menciones_negativas': 0, 'ratio_evidencia': 1.0, 'clasificacion': 'Neutro/Tecnológico'}


In [34]:
import requests
import xml.etree.ElementTree as ET
import re
import time

def analizar_aditivo_pubmed_robusto(aditivo, max_papers=1000):
    # 1. DICCIONARIOS DE KEYWORDS (SENTIMIENTO)
    keywords_riesgo = [
        'toxic', 'toxicity', 'cytotoxicity', 'genotoxicity', 'mutagenic', 'carcinogenic',
        'inflammation', 'pro-inflammatory', 'oxidative stress', 'cytokine', 'interleukin',
        'endocrine disruptor', 'hormonal interference', 'metabolic syndrome', 'obesogenic',
        'insulin resistance', 'hyperglycemia', 'dyslipidemia', 'allergen', 'hypersensitivity', 
        'intolerance', 'anaphylaxis', 'immunogenicity', 'hepatotoxic', 'nephrotoxic', 
        'neurotoxic', 'gastrointestinal distress', 'risk factor', 'adverse effect', 
        'lethal dose', 'noael', 'loael', 'chronic exposure'
    ]
    
    keywords_beneficio = [
        'antioxidant', 'free radical scavenger', 'protective', 'cytoprotective', 'neuroprotective',
        'cardioprotective', 'chemopreventive', 'essential', 'vitamin', 'micronutrient', 
        'bioavailable', 'bioactive', 'nutrient', 'prebiotic', 'probiotic', 'gut microbiota', 
        'microbiome health', 'digestibility', 'anti-inflammatory', 'hypoglycemic', 
        'lipid-lowering', 'metabolic health', 'safe', 'gras', 'therapeutic', 'benefit', 
        'healthy', 'fortification', 'enrichment'
    ]

    # 2. DICCIONARIOS DE CONTEXTO (NLP AVANZADO)
    ctx_humano = ['human', 'patient', 'clinical', 'volunteer', 'men', 'women', 'clinical trial', 'meta-analysis']
    ctx_animal = ['rat', 'mouse', 'murine', 'rabbit', 'animal model', 'in vivo']
    ctx_seguridad = ['safe', 'no adverse', 'not toxic', 'non-toxic', 'well tolerated', 'no significant risk']
    ctx_alerta = ['warning', 'prohibited', 'banned', 'carcinogen', 'disruptor', 'high risk']

    base_url_search = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    base_url_fetch = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    
    query = f'"{aditivo}" AND (health OR safety OR toxicity OR benefit)'
    
    try:
        # BÚSQUEDA DE IDs
        search_res = requests.get(base_url_search, params={"db": "pubmed", "term": query, "retmode": "json", "retmax": max_papers})
        id_list = search_res.json().get("esearchresult", {}).get("idlist", [])
        total_en_pubmed = int(search_res.json().get("esearchresult", {}).get("count", 0))
        
        if not id_list:
            return {"total": total_en_pubmed, "analizados": 0, "ratio": 1.0}

        # VARIABLES ACUMULADORAS
        c_riesgo, c_beneficio = 0, 0
        articulos_leidos = 0
        puntos_evidencia = 0
        c_seguridad, c_alerta = 0, 0
        
        chunk_size = 50 
        
        for i in range(0, len(id_list), chunk_size):
            chunk = id_list[i : i + chunk_size]
            fetch_res = requests.get(base_url_fetch, params={"db": "pubmed", "id": ",".join(chunk), "retmode": "xml"})
            root = ET.fromstring(fetch_res.content)
            
            articulos_en_lote = root.findall(".//PubmedArticle")
            articulos_leidos += len(articulos_en_lote)
            
            for article in articulos_en_lote:
                titulo = (article.find(".//ArticleTitle").text or "").lower()
                abstract_elems = article.findall(".//AbstractText")
                abstract = " ".join([p.text for p in abstract_elems if p.text is not None]).lower()
                texto_full = titulo + " " + abstract
                
                # 1. Conteo de sentimiento
                c_riesgo += len(re.findall(r'\b(' + '|'.join(keywords_riesgo) + r')', texto_full))
                c_beneficio += len(re.findall(r'\b(' + '|'.join(keywords_beneficio) + r')', texto_full))
                
                # 2. Conteo de contexto de seguridad
                c_seguridad += sum(1 for w in ctx_seguridad if w in texto_full)
                c_alerta += sum(1 for w in ctx_alerta if w in texto_full)

                # 3. Clasificación de Nivel de Evidencia (Feature para ML)
                if any(w in texto_full for w in ctx_humano):
                    puntos_evidencia += 3 # Humano / Meta-análisis
                elif any(w in texto_full for w in ctx_animal):
                    puntos_evidencia += 2 # Animal / In vivo
                else:
                    puntos_evidencia += 1 # In vitro / Genérico
            
            print(f"Lote procesado: {i + len(chunk)} / {len(id_list)} artículos...")
            time.sleep(0.4) 

        # MÉTRICAS FINALES
        ratio = (c_beneficio + 1) / (c_riesgo + 1)
        nivel_evidencia_avg = round(puntos_evidencia / articulos_leidos, 2) if articulos_leidos > 0 else 0
        
        return {
            "total_pubmed": total_en_pubmed,
            "analizados": articulos_leidos,
            "ratio_final": round(ratio, 2),
            "nivel_evidencia": nivel_evidencia_avg,
            "menciones_alerta": c_alerta,
            "menciones_seguridad": c_seguridad,
            "intensidad_debate": round((c_riesgo + c_beneficio) / articulos_leidos, 2)
        }

    except Exception as e:
        return f"Error en el flujo: {e}"

# PRUEBA DE FUEGO
print(analizar_aditivo_pubmed_robusto("Aspartame", max_papers=100))

Lote procesado: 50 / 100 artículos...
Lote procesado: 100 / 100 artículos...
{'total_pubmed': 793, 'analizados': 99, 'ratio_final': 1.01, 'nivel_evidencia': 2.75, 'menciones_alerta': 11, 'menciones_seguridad': 36, 'intensidad_debate': 3.0}


In [35]:
def extraer_features_nlp(texto_full):
    # Diccionarios de Contexto
    # 1. Sujeto de estudio (Peso en la jerarquía científica)
    contexto_humano = ['human', 'patient', 'clinical', 'volunteer', 'men', 'women', 'clinical trial', 'meta-analysis']
    contexto_animal = ['rat', 'mouse', 'murine', 'rabbit', 'animal model', 'in vivo']
    contexto_celular = ['in vitro', 'cell culture', 'petri', 'assay', 'cytotoxicity']

    # 2. Contextos de Seguridad (Negación y Estabilidad)
    seguridad = ['safe', 'no adverse', 'not toxic', 'non-toxic', 'well tolerated', 'no significant risk']
    alerta = ['warning', 'prohibited', 'banned', 'carcinogen', 'disruptor', 'high risk']

    # --- Lógica de Extracción ---
    texto = texto_full.lower()
    
    # Determinamos el "Sujeto Dominante"
    score_humano = sum(1 for w in contexto_humano if w in texto)
    score_animal = sum(1 for w in contexto_animal if w in texto)
    
    if score_humano > 0:
        sujeto = "Human"
        nivel_evidencia = 3 # Nivel más alto para el Árbol de Decisión
    elif score_animal > 0:
        sujeto = "Animal"
        nivel_evidencia = 2
    else:
        sujeto = "In Vitro / Genérico"
        nivel_evidencia = 1

    # Detección de Alertas vs Seguridad
    menciones_seguridad = sum(1 for w in seguridad if w in texto)
    menciones_alerta = sum(1 for w in alerta if w in texto)

    return {
        "sujeto": sujeto,
        "nivel_evidencia": nivel_evidencia,
        "menciones_seguridad": menciones_seguridad,
        "menciones_alerta": menciones_alerta
    }

In [2]:
import pandas as pd
import re

# 1. Cargar el archivo
df = pd.read_csv('maestro_aditivos.csv') #

# 2. Definir el patrón de búsqueda: E + (1 a 7 caracteres alfanuméricos)
# El ancla ^ y $ asegura que toda la cadena cumpla la longitud, no solo una parte.
patron_e = r'^E[0-9a-zA-Z]{1,7}$'

# 3. Separar los datos
# Creamos una máscara booleana para filtrar
mask_cumple = df['nombre'].str.split(' - ').str[0].str.match(patron_e).fillna(False)

df_limpio = df[mask_cumple].copy()
df_descartes = df[~mask_cumple].copy()

# 4. Crear el diccionario de descartes
# Esto servirá para ver qué estamos quitando (extractos, gomas con nombres largos, etc.)
diccionario_descartes = df_descartes.set_index('id')['nombre'].to_dict()

# 5. Guardar el maestro final para los 10.000 productos
df_limpio.to_csv('maestro_enriquecido.csv', index=False)

print(f"✅ Aditivos válidos: {len(df_limpio)}")
print(f"❌ Elementos movidos al diccionario de descartes: {len(df_descartes)}")

✅ Aditivos válidos: 646
❌ Elementos movidos al diccionario de descartes: 35
